# Encoding Ordinal Data

### Definition
Ordinal data is a type of **categorical data** where the categories have a **meaningful, ranked order** — but the *distance* between ranks is not necessarily equal or measurable.

### Key Idea
You can say one category is "greater than" or "less than" another, but you can't do meaningful arithmetic on the raw labels (e.g. "High − Medium" doesn't mean anything numerically, even though High > Medium makes sense).

### Examples
- Education level: `High School < Bachelor's < Master's < PhD`
- Satisfaction rating: `Poor < Fair < Good < Excellent`
- Priority: `Low < Medium < High < Critical`
- Clothing size: `S < M < L < XL`

### Where It Sits in the Data Hierarchy

```
Data
├── Categorical
│   ├── Nominal   (no order — e.g. colors, cities)
│   └── Ordinal   (has order — e.g. ratings, sizes)
└── Numerical
```


### How to Identify It
Ask: *"If I sorted these categories, would the order mean something real, or would it be arbitrary?"*
- Ordinal → order is real and informative (Small vs Large clearly differ in magnitude direction).
- Nominal → order would be arbitrary (Red vs Blue — sorting alphabetically means nothing).

### Why It Matters for Encoding
Because the order is meaningful, you're allowed — and expected — to encode ordinal data as **increasing integers that preserve rank** (Ordinal Encoding). Treating it as nominal (e.g. One-Hot Encoding) throws away real information; treating it as pure numerical without validating equal spacing can overstate precision that doesn't exist.

### Gotchas
- Just because it *looks* numeric-adjacent doesn't mean the spacing is equal — "Fair" to "Good" may not be the same jump as "Good" to "Excellent."
- Don't confuse ordinal with **interval/ratio numerical data** — ordinal only guarantees order, not consistent distance.

---

# 1. Ordinal Encoding 



### Definition
Ordinal Encoding converts categories into integers **based on a rank we explicitly define**, preserving the natural order between them (e.g. `Low < Medium < High`).

### Key Idea
The order is not inferred from the data — we need to supply it manually, because only *we* know the real-world ranking the categories represent.

### When to Use
- The categorical feature has a **genuine, meaningful order** (education level, size, satisfaction rating, priority).
- You're encoding an **input feature** for models that can exploit ordering (tree-based models, linear models where the numeric gap is meaningful).

### When NOT to Use
- The categories have **no inherent order** (e.g. colors, cities) — using ordinal encoding here fabricates a false relationship (e.g. implying `Red < Blue`).
- Don't use it blindly on the **target variable** — that's Label Encoding's job.

### Gotchas
- Unseen categories at transform time will break the lookup unless explicitly handled.
- The numeric gaps (0,1,2...) imply equal spacing between ranks, which may not reflect reality (Medium might be "closer" to High than to Low).

### Note
- Always fit the encoder on your training data before applying it to new data. Be sure to use the same encoder on both training and test sets.

- If used different fit on traing and test sets, you'll get different results. This is because ordinal encoding relies on the order of categories to create numeric gaps. If you use different orderings on training and test sets, the resulting numeric gaps will be different. This can lead to unexpected results.

-  If you're using ordinal encoding on a categorical variable that has no inherent order (e.g. colors), it's better to use Label Encoding instead.

In [1]:
# ============================
# MODULE 1: Ordinal Encoding
# ============================

def fit_ordinal_encoder(order):
    
    """
    order : list defining rank e.g. [low, medium, high]
    Returns a mapping dict : {category : rank}
    
    """
    
    mapping={}
    for rank in range(len(order)):
        category = order[rank]
        mapping[category]=rank
    
    return mapping 

def transform_ordinal_data(data, mapping):
    
    encoded = []
    for val in data:
        encoded.append(mapping[val])
    return encoded

def inverse_transform_ordinal(encoded, mapping):
    # flip the mapping : {rank : category}
    
    inverse_mapping={}
    
    for category, rank in mapping.items():
        inverse_mapping[rank]= category    
        
    
    decoded=[] 
    for val in encoded:
        decoded.append(inverse_mapping[val])
    return decoded

# Dummy Examples 1: 
data = ['Medium', 'Low', 'High', 'Low', 'High', 'Medium']

# Define Mapping
mapping = fit_ordinal_encoder(order = ['Low', 'Medium', 'High'])
print(f"Required Order : {mapping}")

# Encoding given data on mapping
encoded_data = transform_ordinal_data(data, mapping) 
print(f"Encoded Data : {encoded_data}")

# Decoding encoded data using mapping
decoded_data = inverse_transform_ordinal(encoded_data, mapping)
print(f"Decoded Data :{decoded_data}")


print("----"*20)
# Dummy Example 2 :
grade = [ 'A', 'B+', 'C', 'D+', 'D', 'B', 'A+', 'C+']

# define Mapping
order=['A+', 'A', 'B+', 'B', 'C+', 'C', 'D+', 'D']
order.reverse()
mapping = fit_ordinal_encoder(order=order)
print(f"Mapping : {mapping}")

# Encoding Given data on mapping
encoded_data = transform_ordinal_data(grade, mapping)
print(f"Encoded Data : {encoded_data}")

# Decoding encoded data on mapping
decoded_data = inverse_transform_ordinal(encoded_data, mapping)
print(f"Decoded Data : {decoded_data}")

Required Order : {'Low': 0, 'Medium': 1, 'High': 2}
Encoded Data : [1, 0, 2, 0, 2, 1]
Decoded Data :['Medium', 'Low', 'High', 'Low', 'High', 'Medium']
--------------------------------------------------------------------------------
Mapping : {'D': 0, 'D+': 1, 'C': 2, 'C+': 3, 'B': 4, 'B+': 5, 'A': 6, 'A+': 7}
Encoded Data : [6, 5, 2, 1, 0, 4, 7, 3]
Decoded Data : ['A', 'B+', 'C', 'D+', 'D', 'B', 'A+', 'C+']


# 2. Label Encoding



### Definition
Label Encoding converts categories into integers with **no implied order** — it's just a consistent, arbitrary integer id per unique category.

### Key Idea
Unlike Ordinal Encoding, the mapping is derived from the data itself (typically sorted alphabetically), not supplied by you — because there's no real ranking to encode.

### When to Use
- Encoding the **target/label variable** in classification problems (this is its intended, canonical use — hence the name).
- Tree-based models can sometimes tolerate it on input features too, since they split on thresholds rather than assuming linear order.

### When NOT to Use
- On **input/independent features** fed to linear models, distance-based models (KNN, SVM), or neural nets — the arbitrary integers imply a false order/magnitude that can mislead the model (e.g. `Fish=2` isn't "more" than `Cat=0`).
- Prefer **One-Hot Encoding** for unordered input features in those cases.

### Gotchas
- Same unseen-category issue as Ordinal Encoding.
- Easy to misuse: the name sounds harmless enough that people apply it to input features by mistake — the *order it creates is meaningless*, so treat it as ids, not magnitudes.
- **Not suitable** for ordinal variables that have a natural order (e.g. `Small=1)



In [2]:
# ============================
# MODULE 2: Label Encoding
# ============================

def fit_label_encoder(data):
    """
    Builds a mapping from unique categories to their integer values.
    assigning the smallest integer value to the first category and incrementing by one for each subsequent category.
    """
    
    unique_values = sorted(set(data))
    mapping={}
    
    for rank in range(len(unique_values)):
        category = unique_values[rank]
        mapping[category] = rank
    return mapping

def transform_label(data, mapping):
    encoded = []    
    for val in data:
        encoded.append(mapping[val])
    return encoded

def inverse_transform_label(encoded, mapping):
    
    inverse_mapping ={}
    for category, rank in mapping.items():
        inverse_mapping[rank]=category
    
    
    decoded =[]        
    for val in encoded:
        decoded.append(inverse_mapping[val])
    return decoded


# Dummy Example :

target = ['Cat', 'Dog', 'Fish', 'chicken']

# Define Mapping
map = fit_label_encoder(target)
print(f"Mapping : {map}")

# Encoding target on map
encoded_target = transform_label(target, map)
print(f"Encoded Target : {encoded_target}")

# Decoding encoded target back to original form
decoded_target = inverse_transform_label(encoded_target, map)
print(f"Decoded Target : {decoded_target}")


Mapping : {'Cat': 0, 'Dog': 1, 'Fish': 2, 'chicken': 3}
Encoded Target : [0, 1, 2, 3]
Decoded Target : ['Cat', 'Dog', 'Fish', 'chicken']


# Using sklearn Library

In [4]:
# ====================================
# MODULE 3: generating Synthetic Data
# ====================================

import pandas as pd
import numpy as np

np.random.seed(42)

# Defining the number of columns 
num_rows = 100

# Defining the Column Names
column_name = ['Age', 'Gender', 'Review', 'Education', 'Purchased']

# Generate Synthetic dataset

data ={
    
    'Age' : np.random.randint(18, 65, size=num_rows),
    'Gender': np.random.choice(['Male', 'Female'], size=num_rows),
    'Review': np.where(np.random.rand(num_rows) < 0.33 , 'Bad',
            np.where(np.random.rand(num_rows) < 0.66, 'Average', 'Good')),
    
    'Education': np.random.choice(['High School', 'Bachelor\'s Degree', 'Master\'s']),
    'purchased': np.random.choice([True, False], size=num_rows)
    
}

# Create dataframe
df = pd.DataFrame(data)

# Save this DataFrame to a CSV file
csv_file_path = "../../data/genrated_data/synthetic_customers.csv"
df.to_csv(csv_file_path, index=False)



In [6]:
# Importing Synthetic data from the generated CSV file

df = pd.read_csv("../../data/genrated_data/synthetic_customers.csv")
df.head()

,Age,Gender,Review,Education,purchased
0,56,Female,Average,High School,False
1,46,Female,Average,High School,True
2,32,Male,Bad,High School,True
3,60,Male,Average,High School,True
4,25,Male,Bad,High School,False


In [7]:
# Extracting columns for encoding
df = df.iloc[:,2:]
df.head()

,Review,Education,purchased
0,Average,High School,False
1,Average,High School,True
2,Bad,High School,True
3,Average,High School,True
4,Bad,High School,False


In [8]:
# ===============================
# MODULE 4: Train Test Splitting
# ===============================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,0:2], df.iloc[:,-1], test_size=0.2, random_state=42)

print(f"Shape of Training set X : {X_train.shape}")
print(f"Shape of Test set X : {X_test.shape}")


Shape of Training set X : (80, 2)
Shape of Test set X : (20, 2)


In [9]:
# =====================================================
# MODULE 5: Applying OrdinalEncoder on Synthetic Data
# =====================================================

from sklearn.preprocessing import OrdinalEncoder

# Initialize the encoder
encoder = OrdinalEncoder(categories=[['Bad', 'Average', 'Good'], ['High School', 'Bachelor\'s Degree', 'Master\'s Degree']])

# Fit on the training data
encoder.fit(X_train)
# NOTE: Always fit on training data and transform both testing data and training data on that fit

# Transform the train and test data
X_train_encoded = encoder.transform(X_train)
X_test_encoded = encoder.transform(X_test)

print(X_test_encoded)



[[1. 0.]
 [0. 0.]
 [1. 0.]
 [1. 0.]
 [0. 0.]
 [2. 0.]
 [0. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [0. 0.]
 [1. 0.]
 [0. 0.]
 [1. 0.]
 [1. 0.]
 [0. 0.]
 [2. 0.]
 [1. 0.]
 [0. 0.]
 [1. 0.]]


In [10]:
# check categories
encoder.categories_

[array(['Bad', 'Average', 'Good'], dtype=object),
 array(['High School', "Bachelor's Degree", "Master's Degree"],
       dtype=object)]

In [11]:
# =====================================================
# MODULE 6: Applying LabelEncoder on Synthetic Data
# =====================================================


from sklearn.preprocessing import LabelEncoder

# Initialize the encoder
label_encoder = LabelEncoder()

# Fit on the training data
label_encoder.fit(y_train)

# Transform the train and test data
y_train_encoded = label_encoder.transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print(y_train_encoded)


[1 1 1 0 0 0 1 1 1 1 1 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 1 1 0 1 1 1 1 1 1 0 1
 0 0 0 0 1 0 1 0 0 0 1 1 0 1 0 1 1 0 0 0 0 1 0 0 1 1 0 1 1 1 1 0 1 1 0 1 1
 1 1 0 0 0 0]
